In [59]:
from sklearn.model_selection import train_test_split
import os
import pandas as pd

In [65]:
from transformers import GPT2TokenizerFast

tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

c:\Users\Aman\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Aman\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:


data_dir = r"C:\Users\Aman\shakespeare\data"

rows = []

for file in os.listdir(data_dir):
    if file.endswith("_original.snt.aligned"):

        play = file.replace("_original.snt.aligned", "")
        original_path = os.path.join(data_dir, file)
        modern_path = os.path.join(
            data_dir,
            play + "_modern.snt.aligned"
        )

        if not os.path.exists(modern_path):
            continue

        with open(original_path, "r", encoding="utf-8") as f:
            original = f.readlines()

        with open(modern_path, "r", encoding="utf-8") as f:
            modern = f.readlines()

        for modern_text, shakespeare_text in zip(modern, original):

            modern_text = modern_text.strip()
            shakespeare_text = shakespeare_text.strip()

            if modern_text and shakespeare_text:

                rows.append({
                    "play": play,
                    "modern": modern_text,
                    "shakespeare": shakespeare_text
                })


df = pd.DataFrame(rows)

print(df.shape)
print(df.head())
print(df["play"].value_counts())

(21079, 3)
                   play                                             modern  \
0  antony-and-cleopatra  I have half a mind to hit you before you speak...   
1  antony-and-cleopatra  But if Antony is alive, healthy, friendly with...   
2  antony-and-cleopatra                                  Madam, he’s well.   
3  antony-and-cleopatra                                That’s well spoken.   
4  antony-and-cleopatra                     And he’s friendly with Caesar.   

                                         shakespeare  
0    I have a mind to strike thee ere thou speak’st.  
1  Yet if thou say Antony lives, is well, Or frie...  
2                                  Madam, he’s well.  
3                                         Well said.  
4                           And friends with Caesar.  
play
othello                 1854
lear                    1722
antony-and-cleopatra    1706
romeojuliet             1462
richardiii              1427
juliuscaesar            1396
hamlet     

In [24]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)

print(dataset)

Dataset({
    features: ['play', 'modern', 'shakespeare'],
    num_rows: 21079
})


In [25]:
dataset[0]

{'play': 'antony-and-cleopatra',
 'modern': 'I have half a mind to hit you before you speak again.',
 'shakespeare': 'I have a mind to strike thee ere thou speak’st.'}

In [26]:
df.head()

,play,modern,shakespeare
0,antony-and-cleopatra,I have half a mind to hit you before you speak...,I have a mind to strike thee ere thou speak’st.
1,antony-and-cleopatra,"But if Antony is alive, healthy, friendly with...","Yet if thou say Antony lives, is well, Or frie..."
2,antony-and-cleopatra,"Madam, he’s well.","Madam, he’s well."
3,antony-and-cleopatra,That’s well spoken.,Well said.
4,antony-and-cleopatra,And he’s friendly with Caesar.,And friends with Caesar.


In [55]:
l=df['play'].unique()

length=[]
for i in l:
    length.append(len(df[df['play']==i]))
average_length=sum(length)/len(length)
average_length

    

1239.9411764705883

In [58]:
train=13
test=2
valid=2


In [71]:

plays = df["play"].unique().tolist()

train_plays, temp_plays = train_test_split(
    plays,
    test_size=4,
    random_state=42
)

val_plays, test_plays = train_test_split(
    temp_plays,
    test_size=2,
    random_state=42
)

print("Train plays:", train_plays)
print("Validation plays:", val_plays)
print("Test plays:", test_plays)
train_df = df[df["play"].isin(train_plays)].copy()

val_df = df[df["play"].isin(val_plays)].copy()

test_df = df[df["play"].isin(test_plays)].copy()

Train plays: ['othello', 'shrew', 'merchant', 'romeojuliet', 'errors', 'msnd', 'twelfthnight', 'henryv', 'macbeth', 'muchado', 'richardiii', 'hamlet', 'lear']
Validation plays: ['antony-and-cleopatra', 'juliuscaesar']
Test plays: ['asyoulikeit', 'tempest']


In [72]:
text = "Wherefore art thou"

tokens = tokenizer.encode(text)

print(tokens)

[8496, 754, 1242, 14210]


In [73]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id

50256

In [78]:
sample = train_df.iloc[0]

print(sample["modern"])
print(sample["shakespeare"])
modern_ids = tokenizer.encode(
    sample["modern"],
    truncation=True,
    max_length=256
)

shakespeare_ids = tokenizer.encode(
    sample["shakespeare"],
    truncation=True,
    max_length=256
)

print(modern_ids)
print(shakespeare_ids)

Time’s not the one in debt.
As if time were in debt.
[7575, 447, 247, 82, 407, 262, 530, 287, 5057, 13]
[1722, 611, 640, 547, 287, 5057, 13]


In [80]:
MAX_LENGTH = 256

train_inputs = tokenizer(
    train_df["modern"].tolist(),
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH
)

train_targets = tokenizer(
    train_df["shakespeare"].tolist(),
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH
)

val_inputs = tokenizer(
    val_df["modern"].tolist(),
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH
)

val_targets = tokenizer(
    val_df["shakespeare"].tolist(),
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH
)

test_inputs = tokenizer(
    test_df["modern"].tolist(),
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH
)

test_targets = tokenizer(
    test_df["shakespeare"].tolist(),
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH
)

In [81]:
import torch
from torch.utils.data import Dataset

class ShakespeareDataset(Dataset):

    def __init__(self, inputs, targets):
        self.input_ids = inputs["input_ids"]
        self.target_ids = targets["input_ids"]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(
                self.input_ids[idx],
                dtype=torch.long
            ),
            "target_ids": torch.tensor(
                self.target_ids[idx],
                dtype=torch.long
            )
        }

In [82]:
train_dataset = ShakespeareDataset(train_inputs, train_targets)
val_dataset = ShakespeareDataset(val_inputs, val_targets)
test_dataset = ShakespeareDataset(test_inputs, test_targets)

In [83]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [85]:
#i need to tune this
d_model = 384
n_heads = 6
num_layers = 4
d_ff = 1536
dropout = 0.1
max_length = 256
vocab_size = 50257